In [1]:
import re
from collections import defaultdict

# 성경 절 포맷 정규 표현식
verse_pattern = re.compile(r'([1-3]?\s?[A-Za-z]+)\s(\d+):(\d+)')

def parse_bible_text(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # 절 시작점 기준으로 분리 (Eph 2:5 같은 패턴을 기준으로)
    matches = list(verse_pattern.finditer(text))

    results = defaultdict(list)
    anomalies = []

    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

        book = match.group(1).strip()
        chapter = match.group(2)
        verse = match.group(3)
        reference = f"{book} {chapter}:{verse}"
        content = text[start:end].strip()

        # 장별로 저장
        chapter_key = f"{book} {chapter}"
        results[chapter_key].append((reference, content))

        # 이상치 탐지: content 내에 또 다른 verse pattern이 있는 경우
        inner_matches = list(verse_pattern.finditer(content[match.end() - start:]))
        if inner_matches:
            anomalies.append((reference, content))

    return results, anomalies


# 예시 사용
file_path = 'ESV-text/ESV_cleaned.txt'  # <- 파일 경로를 여기에 넣으세요
chapter_data, anomalies = parse_bible_text(file_path)

# 출력
# print("=== 이상치 감지된 절들 ===")
# for ref, text in anomalies:
#     print(f"[{ref}]: {text}\n")
total = 0

print("\n=== 장별 구절 수 ===")
for chapter, verses in chapter_data.items():
    if len(chapter.split(' ')[0]) > 4:
        print(f"{chapter}: {len(verses)}절")
        total += 1

print(f"전체 {total}개")


=== 장별 구절 수 ===
전체 0개


In [28]:
import re
from collections import defaultdict

# 성경 절 포맷 정규 표현식 (약어용)
verse_pattern = re.compile(r'([1-3]?[A-Za-z]{2,})\s(\d+):(\d+)')

def parse_bible_text(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()

    matches = list(verse_pattern.finditer(text))
    results = defaultdict(list)
    anomalies = []

    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

        book = match.group(1).strip()
        chapter = match.group(2)
        verse = match.group(3)
        reference = f"{book} {chapter}:{verse}"
        content = text[start:end].strip()

        # 장별로 저장
        chapter_key = f"{book} {chapter}"
        results[chapter_key].append((reference, content))

        # 이상치 탐지: content 내에 또 다른 verse pattern이 있는 경우
        inner_matches = list(verse_pattern.finditer(content[match.end() - start:]))
        if inner_matches:
            anomalies.append((reference, content))

    return results, anomalies


# 예시 사용
file_path = 'ESV-text/ESV_cleaned.txt'
chapter_data, anomalies = parse_bible_text(file_path)

# 출력
total_chapter = 0
total_verse = 0
chapters_list = []
verses_list = []

print("\n=== 장별 구절 수 ===")
for chapter, verses in chapter_data.items():
    if len(chapter.split(' ')[0]) > 2:  # 약어 기준
        print(f"{chapter}: {len(verses)}절")
        total_chapter += 1
        total_verse += len(verses)
        chapters_list.append(chapter)
        verses_list.append(len(verses))

print(f"전체 {total_chapter}개")
print(f"전체 {total_verse}개")



=== 장별 구절 수 ===
Gen 1: 31절
Gen 2: 25절
Gen 3: 24절
Gen 4: 26절
Gen 5: 32절
Gen 6: 22절
Gen 7: 24절
Gen 8: 22절
Gen 9: 29절
Gen 10: 32절
Gen 11: 32절
Gen 12: 20절
Gen 13: 18절
Gen 14: 24절
Gen 15: 21절
Gen 16: 16절
Gen 17: 27절
Gen 18: 33절
Gen 19: 38절
Gen 20: 18절
Gen 21: 34절
Gen 22: 24절
Gen 23: 20절
Gen 24: 67절
Gen 25: 34절
Gen 26: 35절
Gen 27: 46절
Gen 28: 22절
Gen 29: 35절
Gen 30: 43절
Gen 31: 55절
Gen 32: 32절
Gen 33: 20절
Gen 34: 31절
Gen 35: 29절
Gen 36: 43절
Gen 37: 36절
Gen 38: 30절
Gen 39: 23절
Gen 40: 23절
Gen 41: 57절
Gen 42: 38절
Gen 43: 34절
Gen 44: 34절
Gen 45: 28절
Gen 46: 34절
Gen 47: 31절
Gen 48: 22절
Gen 49: 33절
Gen 50: 26절
Exo 1: 22절
Exo 2: 25절
Exo 3: 22절
Exo 4: 31절
Exo 5: 23절
Exo 6: 30절
Exo 7: 25절
Exo 8: 32절
Exo 9: 35절
Exo 10: 29절
Exo 11: 10절
Exo 12: 51절
Exo 13: 22절
Exo 14: 31절
Exo 15: 27절
Exo 16: 36절
Exo 17: 16절
Exo 18: 27절
Exo 19: 25절
Exo 20: 26절
Exo 21: 36절
Exo 22: 31절
Exo 23: 33절
Exo 24: 18절
Exo 25: 40절
Exo 26: 37절
Exo 27: 21절
Exo 28: 43절
Exo 29: 46절
Exo 30: 38절
Exo 31: 18절
Exo 32: 35절
Exo 33: 23절
Exo 3

In [29]:
import os
import sys

# 실행 경로 얻기 (PyInstaller 환경과 일반 환경 모두 지원)
def resource_path(relative_path):
    if hasattr(sys, '_MEIPASS'):
        # PyInstaller 임시폴더 경로
        base_path = sys._MEIPASS
    else:
        base_path = os.path.abspath(".")
    return os.path.join(base_path, relative_path)

def absolute_path(file_path):
    # 절대경로가 이미 주어졌으면 그대로 반환, 아니면 현재 경로 기준으로 반환
    if os.path.isabs(file_path):
        return file_path
    else:
        return os.path.abspath(file_path)

def read_files_in_directory(directory):
    file_contents = []
    for filename in os.listdir(directory):
        if filename.endswith('.txt'):
            with open(os.path.join(directory, filename), 'r', encoding='utf-8') as file:
                content = file.read()
                file_contents.append(content)
    return file_contents

bible_books = [
    "창세기", "출애굽기", "레위기", "민수기", "신명기", "여호수아", "사사기", "룻기", "사무엘상", "사무엘하",
    "열왕기상", "열왕기하", "역대상", "역대하", "에스라", "느헤미야", "에스더", "욥기", "시편", "잠언",
    "전도서", "아가", "이사야", "예레미야", "예레미야애가", "에스겔", "다니엘", "호세아", "요엘", "아모스",
    "오바댜", "요나", "미가", "나훔", "하박국", "스바냐", "학개", "스가랴", "말라기", "마태복음", "마가복음",
    "누가복음", "요한복음", "사도행전", "로마서", "고린도전서", "고린도후서", "갈라디아서", "에베소서", "빌립보서", "골로새서",
    "데살로니가전서", "데살로니가후서", "디모데전서", "디모데후서", "디도서", "빌레몬서", "히브리서", "야고보서", "베드로전서", "베드로후서",
    "요한일서", "요한이서", "요한삼서", "유다서", "요한계시록"
]

texts = read_files_in_directory(resource_path('개역개정-text'))
bible_dict = {bible_books[i]: texts[i] for i in range(len(bible_books))}

In [30]:
def split_and_format_verses(bible_dict):
    result = {}
    for book, verses in bible_dict.items():
        chapter_map = defaultdict(list)
        for verse in verses.splitlines():
            match = re.match(r'([가-힣]+)(\d+):(\d+)\s+(.*)', verse)
            if match:
                chapter = match.group(2)
                verse_num = match.group(3)
                content = match.group(4)
                chapter_map[chapter].append(f"{verse_num} {content}")
        sorted_chapters = sorted(chapter_map.items(), key=lambda x: int(x[0]))
        chapter_list = [verses for _, verses in sorted_chapters]
        result[book] = chapter_list
    return result

formatted_bible = split_and_format_verses(bible_dict)

In [31]:
chapters_list_kor = []
verses_list_kor = []

for book in formatted_bible.keys():
    for chapter in range(len(formatted_bible[book])):
        print(f"{book} {chapter + 1}: {len(formatted_bible[book][chapter])}절")
        chapters_list_kor.append(f"{book} {chapter + 1}")
        verses_list_kor.append(len(formatted_bible[book][chapter]))

창세기 1: 31절
창세기 2: 25절
창세기 3: 24절
창세기 4: 26절
창세기 5: 32절
창세기 6: 22절
창세기 7: 24절
창세기 8: 22절
창세기 9: 29절
창세기 10: 32절
창세기 11: 32절
창세기 12: 20절
창세기 13: 18절
창세기 14: 24절
창세기 15: 21절
창세기 16: 16절
창세기 17: 27절
창세기 18: 33절
창세기 19: 38절
창세기 20: 18절
창세기 21: 34절
창세기 22: 24절
창세기 23: 20절
창세기 24: 67절
창세기 25: 34절
창세기 26: 35절
창세기 27: 46절
창세기 28: 22절
창세기 29: 35절
창세기 30: 43절
창세기 31: 55절
창세기 32: 32절
창세기 33: 20절
창세기 34: 31절
창세기 35: 29절
창세기 36: 43절
창세기 37: 36절
창세기 38: 30절
창세기 39: 23절
창세기 40: 23절
창세기 41: 57절
창세기 42: 38절
창세기 43: 34절
창세기 44: 34절
창세기 45: 28절
창세기 46: 34절
창세기 47: 31절
창세기 48: 22절
창세기 49: 33절
창세기 50: 26절
출애굽기 1: 22절
출애굽기 2: 25절
출애굽기 3: 22절
출애굽기 4: 31절
출애굽기 5: 23절
출애굽기 6: 30절
출애굽기 7: 25절
출애굽기 8: 32절
출애굽기 9: 35절
출애굽기 10: 29절
출애굽기 11: 10절
출애굽기 12: 51절
출애굽기 13: 22절
출애굽기 14: 31절
출애굽기 15: 27절
출애굽기 16: 36절
출애굽기 17: 16절
출애굽기 18: 27절
출애굽기 19: 25절
출애굽기 20: 26절
출애굽기 21: 36절
출애굽기 22: 31절
출애굽기 23: 33절
출애굽기 24: 18절
출애굽기 25: 40절
출애굽기 26: 37절
출애굽기 27: 21절
출애굽기 28: 43절
출애굽기 29: 46절
출애굽기 30: 38절
출애굽기 31: 18절
출애굽기 32: 35절
출애

In [32]:
# formatted_bible의 전체 절 수
total_verses = sum(len(verses) for chapters in formatted_bible.values() for verses in chapters)
print(f"전체 절 수: {total_verses}")

전체 절 수: 31102


In [33]:
import pandas as pd
import numpy as np

chapters_arr = np.array(chapters_list)
verses_arr = np.array(verses_list)

df = pd.DataFrame({
    0: chapters_arr,
    1: verses_arr
})

In [34]:
chapters_arr_kor = np.array(chapters_list_kor)
verses_arr_kor = np.array(verses_list_kor)

df_kor = pd.DataFrame({
    0: chapters_arr_kor,
    1: verses_arr_kor
})

In [35]:
df.to_csv('output_esv.csv', index=False, header=False)
df_kor.to_csv('output_kor.csv', index=False, header=False, encoding='ANSI')